In [1]:
import pandas as pd 

In [2]:
agents = pd.read_csv("../data/raw/agents.csv")
agents.head()

,agent_id,commune,opened_date,agent_type,float_limit
0,AG0126,Selembao,2024-04-07,supermarket,5000
1,AG0058,Kalamu,2024/09/28,kiosk,500
2,AG0283,Selembao,2024-06-27,shop,500
3,AG0067,Barumbu,2024-12-23,NaN,1000
4,AG0115,Masina,2026-01-16,shop,1000


In [3]:
print("rows:", len(agents))
print("distinct agent_id:", agents["agent_id"].nunique())
# Watching how much duplicates and unique values we have.

rows: 303
distinct agent_id: 300


In [4]:
print("missing agent_id:", agents["agent_id"].isna().sum())
print("duplicated agent_id:", agents["agent_id"].duplicated().sum())

missing agent_id: 0
duplicated agent_id: 3


In [5]:
dup_mask = agents["agent_id"].duplicated(keep=False)
agents[dup_mask].sort_values("agent_id")

,agent_id,commune,opened_date,agent_type,float_limit
18,AG0056,Masina,2026-06-21,kiosk,1000
38,AG0056,Masina,2026-06-21,kiosk,1000
176,AG0084,Lingwala,2025-05-30,shop,2000
243,AG0084,Lingwala,2025-05-30,shop,2000
117,AG0206,Limete,2025-05-03,shop,"$2,000"
213,AG0206,Limete,2025-05-03,shop,"$2,000"


In [6]:
dup_mask

0      False
1      False
2      False
3      False
4      False
       ...  
298    False
299    False
300    False
301    False
302    False
Name: agent_id, Length: 303, dtype: bool

## Explanation of keep=False

In [7]:
ids = pd.Series(["A1", "A2", "A2", "A3", "A1"])
ids.duplicated()

0    False
1    False
2     True
3    False
4     True
dtype: bool

In [8]:
ids.duplicated(keep=False)

0     True
1     True
2     True
3    False
4     True
dtype: bool

In [9]:
agents["agent_id"].duplicated(keep=False)

0      False
1      False
2      False
3      False
4      False
       ...  
298    False
299    False
300    False
301    False
302    False
Name: agent_id, Length: 303, dtype: bool

In [10]:
dup_mask = agents["agent_id"].duplicated(keep=False)
agents[dup_mask].sort_values("agent_id")

,agent_id,commune,opened_date,agent_type,float_limit
18,AG0056,Masina,2026-06-21,kiosk,1000
38,AG0056,Masina,2026-06-21,kiosk,1000
176,AG0084,Lingwala,2025-05-30,shop,2000
243,AG0084,Lingwala,2025-05-30,shop,2000
117,AG0206,Limete,2025-05-03,shop,"$2,000"
213,AG0206,Limete,2025-05-03,shop,"$2,000"


In [11]:
agents['float_limit'].str.contains('$').sum()

np.int64(303)

In [12]:
agents['float_limit'].str.contains('$', regex=False).sum()

np.int64(23)

In [13]:
agents["float_limit"].value_counts(dropna=False) 
# value_counts() is like groupby in SQL.

float_limit
500       105
1000       91
2000       49
5000       35
$2,000     10
$1,000      7
$500        5
$5,000      1
Name: count, dtype: int64

In [14]:
agents['agent_type'].value_counts(dropna=False)


agent_type
shop           144
kiosk           92
pharmacy        31
supermarket     28
NaN              8
Name: count, dtype: int64

In [15]:
agents['commune'].value_counts(dropna=False)

commune
Limete         31
Ngaliema       24
Masina         23
Matete         22
Kalamu         21
Lemba          21
Ndjili         21
Selembao       18
Kintambo       18
Gombe          18
Kimbanseke     18
Bandalungwa    15
Barumbu        12
Kasa-Vubu      12
Lingwala       11
Limeté          3
KIMBANSEKE      3
MASINA          3
ngaliema        2
Kimbanseké      2
Ngaliéma        2
Ndjili          1
NGALIEMA        1
LIMETE          1
Name: count, dtype: int64

In [16]:
agents['agent_id'].nunique()

300

In [17]:
agents["commune"].unique()

<StringArray>
[   'Selembao',      'Kalamu',     'Barumbu',      'Masina',    'Lingwala',
      'Limete',       'Lemba',      'Ndjili', 'Bandalungwa',    'Ngaliema',
    'Kintambo',       'Gombe',  'Kimbanseke',      'Limeté',   'ngaliema ',
   'Kasa-Vubu',      'Matete',  'Kimbanseké',  'KIMBANSEKE',      'MASINA',
    'Ngaliéma',     'Ndjili ',    'NGALIEMA',      'LIMETE']
Length: 24, dtype: str

In [18]:
agents

,agent_id,commune,opened_date,agent_type,float_limit
0,AG0126,Selembao,2024-04-07,supermarket,5000
1,AG0058,Kalamu,2024/09/28,kiosk,500
2,AG0283,Selembao,2024-06-27,shop,500
3,AG0067,Barumbu,2024-12-23,NaN,1000
4,AG0115,Masina,2026-01-16,shop,1000
...,...,...,...,...,...
298,AG0212,Masina,2025/10/16,shop,500
299,AG0068,Kintambo,2025-02-28,kiosk,2000
300,AG0026,Selembao,2025-06-21,shop,1000
301,AG0197,Masina,30-09-2024,kiosk,5000


## Data problems found in `agents`

- **Grain and key:** one row = one agent; the key is `agent_id`.
- **Duplicates:** 303 rows but only 300 distinct `agent_id`. The 3 extra rows are exact copies, so they are safe to remove.
- **`agent_type`:** 8 of 303 values are missing (NaN). The 4 types (shop, kiosk, pharmacy, supermarket) are spelled consistently.
- **`float_limit`:** only 4 real amounts (5000, 2000, 1000, 500), but 23 values are written with `$` and 18 of them also have a thousands separator (e.g. `$5,000`). No missing values.
- **`commune`:** 24 different spellings but only 15 real communes. 18 rows are misspelled: upper/lower case (e.g. `KIMBANSEKE`), accents (e.g. `Limeté`), trailing spaces (e.g. `Ndjili `). Only 15 of Kinshasa's 24 communes appear.
- **`opened_date`:** 4 date formats, no missing values: `YYYY-MM-DD` (252 rows), `YYYY/MM/DD` (18), `DD-MM-YYYY` (18) and `DD/MM/YYYY` (15). Day-first is proven by dates like `24/05/2026` and `27-07-2024`, and no date has a second number above 12. 10 dates remain ambiguous on their own (e.g. `03/04/2025`); we assume they follow the same order.
- **Open questions:** the currency of `float_limit` (probably USD, to confirm), and the exact meaning of `float_limit`.

# Section 2: Date Analysis

In [19]:
agents.head()

,agent_id,commune,opened_date,agent_type,float_limit
0,AG0126,Selembao,2024-04-07,supermarket,5000
1,AG0058,Kalamu,2024/09/28,kiosk,500
2,AG0283,Selembao,2024-06-27,shop,500
3,AG0067,Barumbu,2024-12-23,NaN,1000
4,AG0115,Masina,2026-01-16,shop,1000


In [27]:
dates = agents['opened_date']
dates = dates.str.replace(r"\d", "9", regex=True)

In [29]:
dates.value_counts(dropna=False)

opened_date
9999-99-99    252
9999/99/99     18
99-99-9999     18
99/99/9999     15
Name: count, dtype: int64

In [30]:
agents["opened_date"][dates == "99/99/9999"]

9      11/03/2024
106    24/05/2026
138    29/12/2025
151    06/01/2026
162    03/04/2025
163    27/06/2026
167    07/01/2026
169    10/04/2024
186    13/07/2025
189    21/04/2025
201    31/05/2024
221    23/03/2026
258    15/11/2024
284    18/11/2024
287    30/06/2025
Name: opened_date, dtype: str

In [31]:
agents["opened_date"][dates == "99-99-9999"]

6      23-01-2025
21     08-08-2024
24     11-10-2024
108    17-06-2024
111    30-01-2026
118    03-11-2025
142    13-09-2025
143    10-01-2025
166    24-08-2024
173    01-07-2024
185    02-04-2024
194    27-07-2024
227    22-05-2025
228    30-01-2025
231    24-02-2026
264    14-05-2026
286    28-04-2024
301    30-09-2024
Name: opened_date, dtype: str